# Обучение базовых моделей Altpay5

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt, ticker as mticker
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.precision', 4)

In [ ]:
PATH = Path("..", "data", "processed", "altpay5_data.csv")
df = pd.read_csv(PATH, sep=";")

DTTM_COLS = [
    'month',
    'first_trx_month_inn',
    'last_active_month_inn',
    'first_month_p1p2',
    'first_month_p3',
]

for col in DTTM_COLS:
    df[col] = pd.to_datetime(df[col])

# Приводим bool-колонки к int8, чтобы они корректно обрабатывались в моделях
bool_cols = df.select_dtypes(include='bool').columns
for col in bool_cols:
    df[col] = df[col].astype('int8')

### Случайный отбор

In [ ]:
k_values = [500, 1000, 1500, 2000, 2500]

target_monthly = df_altpay5.groupby('month').target.sum()
month_sizes = df_altpay5.groupby('month').size()

precision = (target_monthly / month_sizes).mean()  # macro average

print("Random Selection P1P2")

for k in k_values:
    coverage = (k / month_sizes).mean()
    expected_conversions = precision * k

    print(
        f"K={k:<4} | "
        f"Precision: {precision:.2%} | "
        f"Coverage: {coverage:.2%} | "
        f"Expected Conversions: {expected_conversions:.2f}"
    )

Random Selection P1P2
K=500  | Precision: 0.14% | Coverage: 1.97% | Expected Conversions: 0.68
K=1000 | Precision: 0.14% | Coverage: 3.94% | Expected Conversions: 1.35
K=1500 | Precision: 0.14% | Coverage: 5.91% | Expected Conversions: 2.03
K=2000 | Precision: 0.14% | Coverage: 7.88% | Expected Conversions: 2.70
K=2500 | Precision: 0.14% | Coverage: 9.85% | Expected Conversions: 3.38


### Business-Rule

Критерии релевантности:
- Отсутствие подключённого продукта `Altpay5`
- Статус `'Active'`, `'Reborn'`
- Только релевантные MCC (флаг `is_relevant_mcc_altpay5`)
- Минимальный порог оборота (флаг `is_relevant_turnover_altpay5`)

In [ ]:
def add_rule_flag_altpay5(df: pd.DataFrame) -> pd.DataFrame:
    active_statuses = ["Active", "Reborn"]

    df = df.copy()
    
    df["rule_flag"] = (
        df["inn_status"].isin(active_statuses)
        & df["is_relevant_mcc_altpay5"]
        & df["is_relevant_turnover_altpay5"]
    ).astype("int8")

    return df


df_altpay5 = add_rule_flag_altpay5(df_altpay5)

In [ ]:
print(f"Business-Rule Selection Altpay5, y_score='turnover'")

calculate_metrics(data=df_altpay5, y_score='turnover', business_rule_mode=True)

Business-Rule Selection Altpay5, y_score='turnover'
K=500  | Precision: 1.00% | Recall: 16.40% | Expected Conversions: 5.00
K=1000 | Precision: 0.90% | Recall: 27.94% | Expected Conversions: 9.00
K=1500 | Precision: 0.79% | Recall: 36.16% | Expected Conversions: 11.80
K=2000 | Precision: 0.76% | Recall: 45.31% | Expected Conversions: 15.20
K=2500 | Precision: 0.74% | Recall: 51.25% | Expected Conversions: 17.29


Пересмотрим метрику ранжирования

In [ ]:
numeric_features = [col for col in df_altpay5.select_dtypes(include=['number']).columns if col not in ('target', 'rule_flag', 'altpay5_turnover')]

k_values = [500, 1000, 1500, 2000, 2500]

print(f"Business-Rule Selection Altpay5")

for asc in (True, False):
    for feature in numeric_features:
        y_score = feature
        print(f"y_score='{y_score}' {'ASC' if asc else 'DESC'}")
        for k in k_values:
            df_k = (
                df_altpay5.query('rule_flag == 1')
                .sort_values(by=y_score, ascending=asc)
                .groupby('month')
                .head(k)
            )

            tp = df_k[df_k['target'] == 1].groupby('month').size()
            fp = df_k[df_k['target'] == 0].groupby('month').size()

            total_positives = (
                df_altpay5[df_altpay5['target'] == 1]
                .groupby('month')
                .size()
            )

            tp, fp = tp.align(fp, fill_value=0)
            tp, total_positives = tp.align(total_positives, fill_value=0)

            precision_k = (tp / (tp + fp)).mean()
            recall_k = (tp / total_positives).mean()

            avg_rows = df_k.groupby('month').size().mean()
            expected_conversions = precision_k * avg_rows

            print(
                f"K={k:<4} | "
                f"Precision: {precision_k:.2%} | "
                f"Recall: {recall_k:.2%} | "
                f"Expected Conversions: {expected_conversions:.2f}"
            )
        print()

Business-Rule Selection P3
y_score='lifetime_month_streak_inn' ASC
K=500  | Precision: 1.49% | Recall: 21.09% | Expected Conversions: 7.47
K=1000 | Precision: 1.14% | Recall: 34.56% | Expected Conversions: 11.40
K=1500 | Precision: 0.94% | Recall: 42.25% | Expected Conversions: 14.07
K=2000 | Precision: 0.78% | Recall: 46.89% | Expected Conversions: 15.53
K=2500 | Precision: 0.74% | Recall: 51.25% | Expected Conversions: 17.29

y_score='altpay1_turnover' ASC
K=500  | Precision: 0.81% | Recall: 10.79% | Expected Conversions: 4.07
K=1000 | Precision: 0.79% | Recall: 21.75% | Expected Conversions: 7.93
K=1500 | Precision: 0.76% | Recall: 31.58% | Expected Conversions: 11.40
K=2000 | Precision: 0.73% | Recall: 40.64% | Expected Conversions: 14.60
K=2500 | Precision: 0.74% | Recall: 51.25% | Expected Conversions: 17.29

y_score='altpay2_turnover' ASC
K=500  | Precision: 0.69% | Recall: 9.22% | Expected Conversions: 3.47
K=1000 | Precision: 0.60% | Recall: 18.29% | Expected Conversions: 6.00

### Logistic Regression

Подготовим данные для обучения логистической регрессии

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df_altpay5_preprocessed = df_altpay5.copy()

df_altpay5_preprocessed['months_since_last_active_month'] = (
    (df_altpay5_preprocessed['month'].dt.year - df_altpay5_preprocessed['last_active_month_inn'].dt.year) * 12 +
    (df_altpay5_preprocessed['month'].dt.month - df_altpay5_preprocessed['last_active_month_inn'].dt.month)
)

df_altpay5_preprocessed["months_since_first_trx"] = (
    (df_altpay5_preprocessed["month"].dt.year - df_altpay5_preprocessed["first_trx_month_inn"].dt.year) * 12 +
    (df_altpay5_preprocessed["month"].dt.month - df_altpay5_preprocessed["first_trx_month_inn"].dt.month)
)

df_altpay5_preprocessed["has_p1p2"] = df_altpay5_preprocessed["first_month_p1p2"].notna()

df_altpay5_preprocessed["has_p3"] = df_altpay5_preprocessed["first_month_p3"].notna()

cols_to_drop = [
    'inn',
    "last_active_month_inn",
    "first_trx_month_inn",
    "top_mcc_group_inn",
    "who_is_this_first_inn",
    "first_month_p1p2",
    "first_month_p3",
]

df_altpay5_preprocessed = (
    df_altpay5_preprocessed
    .sort_values(by='month')
    .reset_index(drop=True)
    .drop(columns=cols_to_drop)
)

scaler = MinMaxScaler()

already_scaled_features = []
for feature in ['turnover', 'cnt_trx', 'avg_check']:
    l = [col for col in df_altpay5_preprocessed.columns if feature in col and not ('change' in col or 'cv' in col or 'is_relevant' in col)]
    already_scaled_features.extend(l)

numeric_cols = df_altpay5_preprocessed.select_dtypes(include='number').columns.tolist()
features_to_scale = [col for col in numeric_cols if col not in already_scaled_features and col not in ['target']]

df_altpay5_preprocessed[features_to_scale] = scaler.fit_transform(df_altpay5_preprocessed[features_to_scale])

df_altpay5_preprocessed = pd.get_dummies(df_altpay5_preprocessed)

In [ ]:
train = df_altpay5_preprocessed.loc[df_altpay5_preprocessed['month'] < '2025-12-01'] # 80%
test = df_altpay5_preprocessed.loc[df_altpay5_preprocessed['month'] >= '2025-12-01'] # 20%

X_train = train.drop(columns=['target', 'month'])
y_train = train['target']

X_test = test.drop(columns=['target', 'month'])
y_test = test['target']

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=100_000)

lr.fit(X_train, y_train)

test_with_lr_proba = test.copy()
train_with_lr_proba = train.copy()

train_with_lr_proba['lr_proba'] = lr.predict_proba(X_train)[:, 1]
test_with_lr_proba['lr_proba'] = lr.predict_proba(X_test)[:, 1]

In [ ]:
print(f"Logistic Regression Altpay5, y_score='lr_proba'\n")

print(f"Train:")
calculate_metrics(data=train_with_lr_proba, y_score='lr_proba')
print()

print(f"Test:")
calculate_metrics(data=test_with_lr_proba, y_score='lr_proba')
print()

Logistic Regression Altpay5, y_score='lr_proba'

Train:
K=500  | Precision: 3.13% | Recall: 38.17% | Expected Conversions: 15.67
K=1000 | Precision: 2.00% | Recall: 50.28% | Expected Conversions: 20.00
K=1500 | Precision: 1.49% | Recall: 55.82% | Expected Conversions: 22.33
K=2000 | Precision: 1.30% | Recall: 65.49% | Expected Conversions: 25.92
K=2500 | Precision: 1.10% | Recall: 69.60% | Expected Conversions: 27.50

Test:
K=500  | Precision: 1.53% | Recall: 44.07% | Expected Conversions: 7.67
K=1000 | Precision: 0.97% | Recall: 55.78% | Expected Conversions: 9.67
K=1500 | Precision: 0.67% | Recall: 57.63% | Expected Conversions: 10.00
K=2000 | Precision: 0.55% | Recall: 63.33% | Expected Conversions: 11.00
K=2500 | Precision: 0.47% | Recall: 67.19% | Expected Conversions: 11.67

